# LLM Alignment with DPO and KTO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/alignment/alignment_dpo.ipynb)

This notebook walks through aligning **Llama-3.1-8B** with human preferences using Ludwig's built-in
**Direct Preference Optimization (DPO)** trainer, then shows how to switch to **KTO** when paired
preference data is not available.

We use the publicly available [Anthropic HH-RLHF](https://huggingface.co/datasets/Anthropic/hh-rlhf)
dataset, which contains helpfulness and harmlessness preference pairs collected from human annotators.

### What you will learn
- How to parse the HH-RLHF dataset into Ludwig's expected column format
- How to configure and run DPO alignment training with LoRA
- How to evaluate response quality before and after alignment
- How to switch to KTO with a one-line config change
- How to upload the aligned model to HuggingFace Hub

### Prerequisites
- **GPU**: A100 (40 GiB) recommended. T4 will work with smaller batch sizes but is slower.
- **HuggingFace token**: Required for Llama-3.1-8B. Set `HUGGING_FACE_HUB_TOKEN` before running.
- **Model access**: Request access at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

## Install dependencies

In [ ]:
!pip install "ludwig[llm]" datasets --quiet

## Setup

In [ ]:
import os

# Set your HuggingFace token — required to download Llama-3.1-8B.
# In Colab: Secrets panel (key icon) -> add HF_TOKEN, then reference it here.
try:
    from google.colab import userdata
    os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # Running locally — export HUGGING_FACE_HUB_TOKEN in your shell instead

assert os.environ.get("HUGGING_FACE_HUB_TOKEN"), (
    "HUGGING_FACE_HUB_TOKEN is not set. "
    "Add it to Colab Secrets or export it in your shell."
)

In [ ]:
import subprocess
import sys

# Verify GPU availability
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode == 0:
    print("GPU(s) detected:")
    print(result.stdout.strip())
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("In Colab: Runtime -> Change runtime type -> A100 GPU")

## Prepare dataset

The [Anthropic HH-RLHF](https://huggingface.co/datasets/Anthropic/hh-rlhf) dataset stores full
multi-turn conversations in two columns: `chosen` (preferred response) and `rejected` (dispreferred
response). Each value is a raw conversation string like:

```
\n\nHuman: How do I make pasta?\n\nAssistant: Boil water, add pasta...\n\nHuman: What sauce?\n\nAssistant: Try marinara...
```

Ludwig's DPO trainer expects three columns: `prompt`, `chosen`, `rejected`, where `prompt` is the
last human turn and `chosen`/`rejected` are the final assistant responses.

The cell below does that extraction.

In [ ]:
import re
import pandas as pd
from datasets import load_dataset


def extract_last_human_turn(conversation: str) -> str:
    """Extract the last Human turn from an HH-RLHF conversation string."""
    turns = re.findall(r"\n\nHuman: (.*?)(?=\n\nAssistant:|\Z)", conversation, re.DOTALL)
    return turns[-1].strip() if turns else conversation.strip()


def extract_last_assistant_turn(conversation: str) -> str:
    """Extract the last Assistant turn from an HH-RLHF conversation string."""
    turns = re.findall(r"\n\nAssistant: (.*?)(?=\n\nHuman:|\Z)", conversation, re.DOTALL)
    return turns[-1].strip() if turns else ""


def prepare_dpo_dataset(split, max_samples=None):
    if max_samples:
        split = split.select(range(min(max_samples, len(split))))
    rows = []
    for ex in split:
        prompt = extract_last_human_turn(ex["chosen"])
        chosen = extract_last_assistant_turn(ex["chosen"])
        rejected = extract_last_assistant_turn(ex["rejected"])
        if prompt and chosen and rejected:
            rows.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return pd.DataFrame(rows)


print("Downloading Anthropic/hh-rlhf ...")
hh = load_dataset("Anthropic/hh-rlhf")
print(f"Train size: {len(hh['train'])}, Test size: {len(hh['test'])}")

In [ ]:
# Use a subset for a quick experiment. Remove max_samples caps for full training.
MAX_TRAIN = 5000
MAX_TEST = 500

train_df = prepare_dpo_dataset(hh["train"], max_samples=MAX_TRAIN)
test_df = prepare_dpo_dataset(hh["test"], max_samples=MAX_TEST)

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

print(f"Saved {len(train_df)} train rows to train.csv")
print(f"Saved {len(test_df)} test rows to test.csv")
train_df.head(2)

In [ ]:
# Preview: show a prompt with its chosen vs rejected response
row = train_df.iloc[0]
print("=== PROMPT ===")
print(row["prompt"])
print("\n=== CHOSEN ===")
print(row["chosen"])
print("\n=== REJECTED ===")
print(row["rejected"])

## DPO training

Direct Preference Optimization (DPO) treats alignment as a classification problem: given a prompt,
prefer `chosen` over `rejected`. The loss function directly optimises the log-ratio between the
policy's probability of the chosen response and the reference model's, without needing an explicit
reward model.

Key hyperparameters:
- `beta`: KL penalty coefficient (0.1 is a typical starting point). Higher values keep the aligned
  model closer to the base model.
- `learning_rate`: 5e-7 is much lower than SFT — the policy already knows how to generate text;
  we're only nudging its preferences.
- `lora r=16, alpha=32`: Standard LoRA settings. Increase `r` for larger models or more complex tasks.

In [ ]:
import logging
import yaml
from ludwig.api import LudwigModel

dpo_config = yaml.safe_load("""
model_type: llm
base_model: meta-llama/Llama-3.1-8B

adapter:
  type: lora
  r: 16
  alpha: 32
  dropout: 0.05

trainer:
  type: dpo
  epochs: 1
  learning_rate: 5.0e-7
  batch_size: 2
  gradient_accumulation_steps: 8
  beta: 0.1

input_features:
  - name: prompt
    type: text

output_features:
  - name: chosen
    type: text

backend:
  type: local
""")

dpo_model = LudwigModel(config=dpo_config, logging_level=logging.INFO)

train_stats, _, output_directory = dpo_model.train(
    dataset="train.csv",
    experiment_name="hh_rlhf_dpo",
    output_directory="results",
)

print(f"\nModel saved to: {output_directory}")

## Evaluate

We compare responses from the base model (no alignment) and the DPO-aligned model on a few
prompts from the test set. You should see the aligned model give more helpful, less evasive answers.

In [ ]:
# Load the base model for comparison
base_config = yaml.safe_load("""
model_type: llm
base_model: meta-llama/Llama-3.1-8B

input_features:
  - name: prompt
    type: text

output_features:
  - name: chosen
    type: text

backend:
  type: local
""")

base_model = LudwigModel(config=base_config, logging_level=logging.WARNING)
_ = base_model.train(dataset="train.csv", experiment_name="baseline", output_directory="results_base")

In [ ]:
eval_prompts = test_df["prompt"].head(3).tolist()
eval_df = pd.DataFrame({"prompt": eval_prompts})

base_preds, _ = base_model.predict(dataset=eval_df)
dpo_preds, _ = dpo_model.predict(dataset=eval_df)

for i, prompt in enumerate(eval_prompts):
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"\n--- Base model ---")
    print(base_preds.iloc[i]["chosen_predictions"])
    print(f"\n--- DPO-aligned model ---")
    print(dpo_preds.iloc[i]["chosen_predictions"])

## KTO alternative

**Kahneman-Tversky Optimization (KTO)** is a good alternative to DPO when you cannot collect
paired preference data. Instead of a (chosen, rejected) pair per prompt, KTO requires only a
single response with a boolean label indicating whether it was desirable.

This makes it easy to use binary user feedback (thumbs up / thumbs down, click-through, etc.)
as training signal.

### Dataset format for KTO

We expand each DPO pair into two rows:

In [ ]:
rows_kto = []
for _, row in train_df.iterrows():
    rows_kto.append({"prompt": row["prompt"], "response": row["chosen"], "label": True})
    rows_kto.append({"prompt": row["prompt"], "response": row["rejected"], "label": False})

train_kto_df = pd.DataFrame(rows_kto)
train_kto_df.to_csv("train_kto.csv", index=False)
print(f"KTO dataset: {len(train_kto_df)} rows")
train_kto_df.head(4)

In [ ]:
kto_config = yaml.safe_load("""
model_type: llm
base_model: meta-llama/Llama-3.1-8B

adapter:
  type: lora
  r: 16
  alpha: 32
  dropout: 0.05

trainer:
  type: kto
  epochs: 1
  learning_rate: 5.0e-7
  batch_size: 2
  gradient_accumulation_steps: 8
  beta: 0.1
  desirable_weight: 1.0
  undesirable_weight: 1.0

input_features:
  - name: prompt
    type: text

output_features:
  - name: response
    type: text

backend:
  type: local
""")

kto_model = LudwigModel(config=kto_config, logging_level=logging.INFO)

_, _, kto_output_dir = kto_model.train(
    dataset="train_kto.csv",
    experiment_name="hh_rlhf_kto",
    output_directory="results_kto",
)

print(f"\nKTO model saved to: {kto_output_dir}")

## Save and upload

After training, upload the aligned model to HuggingFace Hub to share it or deploy it to an endpoint.

In [ ]:
# Replace with your HuggingFace org/username and desired repo name
HF_REPO = "your-username/llama-3.1-8b-dpo-hh-rlhf"

# Upload via CLI:
print(f"ludwig upload hf_hub -r {HF_REPO} -m {output_directory}")

# Or via Python API:
# LudwigModel.upload_to_hf_hub(HF_REPO, output_directory)

In [ ]:
# Uncomment to actually run the upload:
# !ludwig upload hf_hub -r {HF_REPO} -m {output_directory}

## Next steps

- **Scale up**: Remove the `max_samples` caps and train on the full HH-RLHF dataset (~160k examples).
- **Try ORPO**: Use `config_orpo.yaml` for single-stage SFT + alignment without a reference model.
- **Iterate on beta**: Lower `beta` (e.g. 0.05) gives more aggressive alignment but risks reward hacking.
- **Evaluation**: Use [MT-Bench](https://github.com/lm-sys/FastChat/tree/main/fastchat/llm_judge) or
  [AlpacaEval](https://github.com/tatsu-lab/alpaca_eval) for rigorous alignment benchmarks.
- **GRPO**: For tasks with a programmatic reward (math, code), see the GRPO trainer docs.